In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
dataset=pd.read_csv("datasets/train.csv")
print(dataset.describe())
print(dataset.head())
dataset["sales_dist"],bins=pd.qcut(dataset['sales'],5,retbins=True)
print(dataset['sales_dist'])
dataset['sales'].hist()
plt.show()
print(dataset['sales_dist'].value_counts(),len(dataset['sales_dist']))

In [ ]:
#Splitting the dataset without bias
dataset = dataset.sort_values(by=['store','item','date'])

split_index = int(len(dataset)*0.8)

train_set = dataset.iloc[:split_index]
test_set  = dataset.iloc[split_index:]

print(train_set['sales_dist'].value_counts()/len(train_set))
print(train_set.head())
train_set.drop('sales_dist',axis=1,inplace=True)
test_set.drop('sales_dist',axis=1,inplace=True)
print(train_set.head())
sales=train_set.copy()
print(sales.head())

In [ ]:
#Feature Engineering

sales['date']=pd.to_datetime(sales['date'])
sales['month']=sales['date'].dt.month
sales['week']=sales['date'].dt.weekday
sales['day']=sales['date'].dt.day
sales['weekday']=sales['week'].isin([5,6]).astype(int)
sales=sales.sort_values(by=['store','item','date'])
print(sales.head())
#Adding lag features:prev_day and mean of week
sales['lag1']=sales.groupby(['store','item'])['sales'].shift(1)
sales['lag7']=sales.groupby(['store','item'])['sales'].shift(7)
sales['rolling7']=sales.groupby(['store','item'])['sales'].shift(1).rolling(7).mean()
print(sales.isnull().sum())
sales.dropna(inplace=True)
print(sales.isnull().sum())



test_df = test_set.copy()

# date
test_df["date"] = pd.to_datetime(test_df["date"])
test_df["month"] = test_df["date"].dt.month
test_df["week"] = test_df["date"].dt.weekday
test_df["day"] = test_df["date"].dt.day
test_df["weekday"] = test_df["week"].isin([5,6]).astype(int)

# sort
test_df = test_df.sort_values(by=["store","item","date"])

# lag
test_df["lag1"] = test_df.groupby(["store","item"])["sales"].shift(1)
test_df["lag7"] = test_df.groupby(["store","item"])["sales"].shift(7)

# rolling
test_df["rolling7"] = test_df.groupby(["store","item"])["sales"].shift(1).rolling(7).mean()

# drop NaN
test_df.dropna(inplace=True)
df=sales

In [ ]:
#Cleaning the Dataset
#hhh
sales_labels=df['sales']
sales=df.drop(['sales','date'],axis=1)
print(sales.isnull().sum())
test_labels=test_df['sales']
test=test_df.drop(['sales','date'],axis=1)
print(test.isnull().sum())
                 


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipelinee=Pipeline([('std_scaler',StandardScaler())])
sales_tr=pipelinee.fit_transform(sales)
test_tr=pipelinee.transform(test)

sales_tr=pd.DataFrame(sales_tr,columns=sales.columns,index=sales.index)
test_tr=pd.DataFrame(test_tr,columns=test.columns,index=test.index)
print(sales_tr.head())
print(test_tr.head())
split = int(len(sales_tr)*0.8)

X_tr = sales_tr[:split]
X_val = sales_tr[split:]

y_tr = sales_labels[:split]
y_val = sales_labels[split:]

In [ ]:
#Training the model:Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,mean_absolute_error
model1=LinearRegression()
model1.fit(X_tr,y_tr)

pred=model1.predict(X_val)
mse=mean_squared_error(y_val,pred)
mae=mean_absolute_error(y_val,pred)
rmse=np.sqrt(mse)
print(rmse)
print(mae)




In [ ]:
#Training the model : Random Forest
from sklearn.ensemble import RandomForestRegressor
model2=RandomForestRegressor(n_estimators=10,
    random_state=42,
    n_jobs=-1,
    max_depth=10,       
    max_features='sqrt',
    min_samples_leaf=4  )
model2.fit(X_tr,y_tr)
pred=model2.predict(X_val)
mse=mean_squared_error(y_val,pred)
mae=mean_absolute_error(y_val,pred)
rmse=np.sqrt(mse)
print(rmse)
print(mae)
import joblib
joblib.dump(model2,"../ML/mlmodel.pkl")